### RAG with Tabalar Data and Vector Memory


In [44]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('Chatbot_rag_v2') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# !pip install langchain
# !pip install langchain_community
# !pip install -qU langchain-ollama
# !pip install python-dotenv
# !pip install -qU langchain-qdrant

In [45]:
import os
from dotenv import load_dotenv

from pyspark.sql.types import StructType
from pyspark.sql import DataFrame
from pyspark.sql.functions import explode, col

In [46]:
%run ./01_Config_env.ipynb

In [47]:
# Variaveis de Ambiente

load_dotenv('./.env')
OLLAMA_API_URL = os.getenv("OLLAMA_API_URL")
token = os.getenv("API_TOKEN")

In [48]:
 %run ./02_Common.ipynb

In [49]:
%run ./03_Get_data.ipynb

Dados disponiveis:
root
 |-- c: string (nullable = true)
 |-- cl: string (nullable = true)
 |-- sl: string (nullable = true)
 |-- lt0: string (nullable = true)
 |-- lt1: string (nullable = true)
 |-- qv: string (nullable = true)
 |-- vs: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- p: string (nullable = true)
 |    |    |-- a: string (nullable = true)
 |    |    |-- ta: string (nullable = true)
 |    |    |-- py: string (nullable = true)
 |    |    |-- px: string (nullable = true)
 |    |    |-- sv: string (nullable = true)
 |    |    |-- is: string (nullable = true)



### Visualizar e pegar uma amostra dos dados

In [50]:
# df = df_posicao.select(
#     col('c').alias('Letreiro_Linha'),
#     col('cl').alias('Linha'),
#     col('sl').alias('Sentido'),
#     col('lt0').alias('Destino_Linha'),
#     col('lt1').alias('Origem_Linha'),
#     col('qv').cast('int').alias('Quantidade_Veiculos')
    
# ).limit(10)

df = df_posicao.select('*').limit(10)

df.createOrReplaceTempView("tbl_bus_posicao")
spark.sql("SELECT * FROM tbl_bus_posicao").show()

+-------+-----+---+--------------------+--------------------+---+--------------------+
|      c|   cl| sl|                 lt0|                 lt1| qv|                  vs|
+-------+-----+---+--------------------+--------------------+---+--------------------+
|2590-10|33680|  2|     PQ. D. PEDRO II|   UNIÃO DE VL. NOVA|  5|[{56327, true, 20...|
|2726-10|  940|  1|         METRÔ PENHA|            LIMOEIRO|  6|[{35990, true, 20...|
|407G-10|34964|  2|        METRÔ CARRÃO|    JD. NOVA VITÓRIA| 10|[{48929, true, 20...|
|208V-10|32975|  2|TERM. PQ. D. PEDR...|TERM. A. E. CARVALHO| 15|[{31150, true, 20...|
|6825-10|   14|  1|     TERM. CAPELINHA|          VALO VELHO|  3|[{72018, true, 20...|
|8050-10|33180|  2|                LAPA|      PQ. MORRO DOCE|  3|[{11034, true, 20...|
|5110-10|34977|  2|       TERM. MERCADO|    TERM. SÃO MATEUS| 10|[{52759, true, 20...|
|1732-10|33439|  2| TERM. AMARAL GURGEL|         VL. SABRINA|  4|[{21755, true, 20...|
|9047-10|  883|  1|                LAPA|   

### Chatbot com RAG

## Funções Auxiliares

In [51]:
# Remover formatação do código gerado
import re

def limpar_sql(resposta_modelo):
    # Remove blocos de código markdown e espaços extras
    sql = re.sub(r"```sql|```", "", resposta_modelo, flags=re.IGNORECASE).strip()
    return sql

In [52]:
def get_metadata(table_name="tbl_bus_posicao"):
    df = spark.sql(f"SELECT * FROM {table_name} LIMIT 1;")
    colunas = "\n".join([f"- {f.name}: {f.dataType.simpleString()}" for f in df.schema])
    return f"Tabela: {table_name}\n\nColunas:\n{colunas}"


## Qdrant Memory

In [53]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="mistral:latest", base_url=OLLAMA_API_URL)

In [54]:
# Cria coleção para armazenar os embeddings 
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
from langchain_core.documents import Document
import uuid

client = QdrantClient(":memory:")

collection_name ="olho_vivo"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=4096, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [76]:
%run ./06.1_Memory.ipynb

In [77]:
qdrant_memory = QdrantMemory(client, embedding)

## Iniciar Mistral 7B

In [57]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="mistral:latest", 
    base_url=OLLAMA_API_URL,
    temperature = 0.3,
 
) 

### Configurar Promps: Roles System e Human

In [58]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt para gerar SQL (Spark)

prompt_sql = ChatPromptTemplate.from_messages([
    ("system", "Você é um especialista em dados. Gere apenas a consulta SQL."),
    ("human", "Contexto:\n{context}\n Estrutura da tabela:\n{schema}\n Pergunta:\n{pergunta}"
     "Escreva uma consulta SQL (somente a SQL) para responder:\n{pergunta}")
])


# Prompt para retornar resultado ao usuario

prompt_resposta = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de dados que responde perguntas com base em resultados SQL."),
    ("human", "Pergunta:\n {pergunta} \n Resultado da consulta:\n {resultado}"
    "Gere uma resposta clara e amigável para o usuário contendo apenas os resultados da consulta.")
])

In [78]:
# Função para gerar resposta com RAG (Tabela + Qdrant)
def resposta_com_rag(pergunta, table_name):
    print("\n💬 Pergunta recebida:", pergunta)

    #Obter metadados da tabela
    schema_txt = get_metadata(table_name)

    # Busca embeddings no Qdrant (Memoria)
    docs = retriever.invoke(pergunta)
    contexto = "\n".join([doc.page_content for doc in docs])

    # Gerar o SQL da query com base na pergunta 
    sql_result = llm.invoke(prompt_sql.format_messages(pergunta=pergunta, schema=schema_txt, context=contexto)).content.strip()
    sql_query=limpar_sql(sql_result)
    print("\n🤖💡 SQL Gerada:", sql_query)

    # Executa query no Spark
    try:
        resultado_df = spark.sql(sql_query)      
        resultado_dict = resultado_df.toPandas().to_dict(orient="records")
    except Exception as e:
        print("\n❌ Erro na execução da SQL:", e)
        return

    # Gera resposta amigável para retornar ao usuario
    resposta = llm.invoke(prompt_resposta.format_messages(pergunta=pergunta, resultado=resultado_dict)).content.strip()
    print("\n🤖 Resposta final:", resposta)

    # Armazena embeddings da pergunta + SQL no Qdrant
    qdrant_memory.ensinar(pergunta, sql_query, metadados={"tabela": table_name, "tipo": "AGG", "schema": schema_txt, "score": 1})
    

    return resposta

In [79]:
resp = resposta_com_rag("Qual o destino (lt0) do viculo do letreiro 1732-10?", "tbl_bus_posicao")


💬 Pergunta recebida: Qual o destino (lt0) do viculo do letreiro 1732-10?

🤖💡 SQL Gerada: SELECT lt0
   FROM tbl_bus_posicao
   WHERE c = '1732-10';

🤖 Resposta final: O destino do viculo do letreiro 1732-10 é a Terminal Amaral Gurgel.

⚠️ Embedding similar já existe. Incrementando score...

Score:  ⭐⭐⭐✩✩


### Listar Exemplos de Consultas

In [80]:
qdrant_memory.listar_exemplos()


🔹 Embedding 1:
Pergunta: Qual o destino (lt0) do viculo do letreiro 1732-10?
SQL: SELECT lt0
FROM tbl_bus_posicao
WHERE c = '1732-10';


## "Ensinar" refinar comportamento manualmente

In [26]:
sql_query =""" SELECT vs.p AS destino
FROM tbl_bus_posicao
WHERE c = '407G-10'"""

In [ ]:
spark.sql(sql_query).show()

In [27]:
table_name = "tbl_bus_posicao"
schema_txt = get_metadata(table_name)

qdrant_memory.ensinar(
    "Qual o destino do viculo do letreiro 407G-10?",
    sql_query, 
    metadados={"tabela": table_name, "tipo": "AGG", "schema": schema_txt, "score": 1}
)

✅ Embedding armazenado na memoria com sucesso!
